In [71]:
import pandas as pd
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor as gpr
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


In [72]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 6

In [73]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

### Treino:
- X → normalização → PCA (fit) → X_pca → GPR (fit)

### Teste:
- X → normalização (transform) → PCA (transform) → X_pca → GPR (predict)

In [74]:
def CreatePCAdf(pca):
    # Matriz de transformação do PCA
    W = pca.components_.T   # shape (n_variaveis, n_componentes)

    # Nomes das componentes
    cp_names = [f"CP{i+1}" for i in range(W.shape[1])]

    # Criar DataFrame
    df_pca = pd.DataFrame(
        data=np.round(W, 3),
        index=PREDICTORS,
        columns=cp_names
    )

    return df_pca    

In [75]:
def TransformPCA(X_train, X_test):
    pca = PCA(n_components=N_COMPONENTS)
    
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca  = pca.transform(X_test)

    print(f"Variância (%): {np.round(pca.explained_variance_ratio_ * 100, 3)}")
    print(f"Total (%): {np.round(np.sum(pca.explained_variance_ratio_) * 100, 3)}")
    df = CreatePCAdf(pca)
    
    return df, pca, X_train_pca, X_test_pca

In [76]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

def plot_train_test_samples(
    y_train, y_train_pred,
    y_test, y_test_pred,
    title="GPR – Treinamento e Teste"
):

    samples_train = np.arange(len(y_train))
    samples_test = np.arange(len(y_test))

    # Figura
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharey=True)

    # -------- Subfigura (a): Treinamento --------
    axes[0].plot(
        samples_train, y_train,
        'o', label="y train (real)", markersize=5
    )
    axes[0].plot(
        samples_train, y_train_pred,
        'x', label="y train (pred)", markersize=5
    )

    axes[0].set_title("(a) Treinamento")
    axes[0].set_xlabel("Amostras")
    axes[0].set_ylabel("Valor")
    axes[0].legend()
    axes[0].grid(True)

    axes[0].text(
        0.02, 0.95,
        f"MSE = {mse_train:.4f}\n$R^2$ = {r2_train:.4f}",
        transform=axes[0].transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.85)
    )

    # -------- Subfigura (b): Teste --------
    axes[1].plot(
        samples_test, y_test,
        'o', label="y test (real)", markersize=5
    )
    axes[1].plot(
        samples_test, y_test_pred,
        'x', label="y test (pred)", markersize=5
    )

    axes[1].set_title("(b) Teste")
    axes[1].set_xlabel("Amostras")
    axes[1].set_ylabel("Valor")
    axes[1].legend()
    axes[1].grid(True)

    axes[1].text(
        0.02, 0.95,
        f"MSE = {mse_test:.4f}\n$R^2$ = {r2_test:.4f}",
        transform=axes[1].transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.85)
    )

    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [77]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

def ComputeMetrics(y_train, y_train_pred, y_test, y_test_pred):
     return {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "r2_train":  r2_score(y_train, y_train_pred),
        "mse_test":  mean_squared_error(y_test, y_test_pred),
        "r2_test":   r2_score(y_test, y_test_pred)
    }

In [78]:
GPR_PARAMS = {
    "Fe": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Al": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "As": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Pb": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Zn": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Hg": {"nu": 0.5, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Co": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e10, "alpha": 1e-3},
    "V":  {"nu": 0.25, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Ba": {"nu": 0.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
    "Mn": {"nu": 1.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
}

In [79]:


def GprModel(X_train_pca, X_test_pca, y_train, y_test, target):

    params = GPR_PARAMS[target]

    kernel = C(1.0) * Matern(
        length_scale=np.ones(X_train_pca.shape[1]),
        nu=params["nu"],
        length_scale_bounds=(params["ls_min"], params["ls_max"])
    )

    model = gpr(
        kernel=kernel,
        alpha=params["alpha"],
        normalize_y=True,
        n_restarts_optimizer=20
    )

   # Treinamento
    model.fit(X_train_pca, y_train)

    # Predições
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Desnormalização
    y_train_denorm = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_train_pred_denorm = OUT_SCALER.inverse_transform(y_train_pred.reshape(-1, 1)).ravel()

    y_test_denorm = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_test_pred_denorm = OUT_SCALER.inverse_transform(y_test_pred.reshape(-1, 1)).ravel()
    
    metrics = ComputeMetrics(y_train_denorm, y_train_pred_denorm,  y_test_denorm, y_test_pred_denorm)
    
    return  metrics

    # plot_train_test_samples(
    #     y_train_denorm, y_train_pred_denorm,
    #     y_test_denorm, y_test_pred_denorm,
    #     title="GPR com Kernel Matern (ν = 0.5)"
    # )

In [ ]:
Results = {}

for i, Dataset in enumerate(Datasets):
    print(f"++++++++++++++++++++++ Pontos {i} ++++++++++++++++++++++++++")

    X = Dataset[PREDICTORS].values
    Y = Dataset[TARGETS].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

    X_train_scaled = SCALER.fit_transform(X_train)
    X_test_scaled  = SCALER.transform(X_test)
    
    df, pca, X_train_pca, X_test_pca = TransformPCA(X_train_scaled, X_test_scaled)
    display(df)

    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")      
        
        y_train = Y_train[:, j]
        y_test  = Y_test[:, j]
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        
        metrics = GprModel(X_train_pca, X_test_pca, y_train, y_test, target)
        
        Results[target] = metrics
        
    display(pd.DataFrame(Results).T)

++++++++++++++++++++++ Pontos 0 ++++++++++++++++++++++++++
Variância (%): [49.822 16.813 11.232  9.747  6.086  4.365]
Total (%): 98.065


,CP1,CP2,CP3,CP4,CP5,CP6
pH,0.385,-0.033,-0.088,0.263,0.193,0.422
Cond,0.439,0.052,-0.059,0.052,-0.087,-0.204
Temp,0.197,0.448,0.240,-0.457,-0.137,0.645
OD,-0.056,-0.262,0.692,0.455,-0.445,0.137
Tds,0.439,0.054,-0.056,0.051,-0.084,-0.199
Resist,-0.434,0.052,-0.094,-0.084,0.097,0.030
Salin,0.434,0.100,-0.044,0.080,-0.075,-0.234
ORP,0.154,-0.218,0.603,-0.465,0.480,-0.288
IP,-0.076,0.505,0.212,0.524,0.595,0.005
Cor,-0.133,0.640,0.172,-0.066,-0.364,-0.409


 → Fe
 → Al


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k2__length_scale is close to the specified upper bound 1000000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k2__length_scale is close to the specified upper bound 1000000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


 → As
 → Pb


...

 → Fe
 → Al
 → As


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


 → Pb
 → Zn
 → Hg
 → Co
 → V
 → Ba
 → Mn


,mse_train,r2_train,mse_test,r2_test
Fe,4.030580e-01,0.999998,191105.663458,0.342293
Al,5.089148e-02,0.999999,2977.638554,-0.516580
As,1.545410e-08,0.999997,0.000729,0.619519
Pb,1.095715e-06,0.999999,0.205896,-16.936011
Zn,4.830613e-04,0.999999,558.980310,-0.228863
Hg,6.024414e-09,0.999999,0.008558,-0.221566
Co,4.020983e-08,0.999998,0.015242,0.271530
V,1.463912e-07,0.999998,0.007298,0.807943
Ba,1.012141e-03,0.999995,28.523983,0.464240
Mn,3.761356e-03,0.999996,1220.566779,0.075777


: 